# Notebook destinado a seleção de variáveis quantitativas para problemas de classificação binária

* Objetivo: **Encontrar um subconjunto de variáveis quantitativas relevantes**  que ajudam a predizer o ALVO/TARGET (**variável binária** de interesse).

## Abordagem de seleção de variáveis

* **Filter Method**: Comparar de forma independente cada variável quantitativa (feature) com o ALVO a fim de quantificar o seu poder preditivo - **perspectiva univariada**.
    * Avalie todas as variáveis $\rightarrow$ escolha um subconjunto $\rightarrow$ aplique um algoritmo de classificação, cluster ou construa regras de negócio para as variáveis de maior poder preditivo.   

## Métricas calculadas

1. **Estatística KS** $\rightarrow$ Mede a diferença máxima entre as distribuições acumuladas de duas amostras (por exemplo, classe 0 versus classe 1 do ALVO). **Quanto maior o valor do KS, mais a variável quantitativa consegue discriminar entre as duas classes do ALVO binário**. É uma métrica eficaz para avaliar o poder de separação de variáveis contínuas.
2. **Fisher Score** $\rightarrow$   Avalia a capacidade de uma feature separar as classes do ALVO. Ele quantifica o quão bem os valores de uma feature se agrupam dentro de cada classe e se separam entre as classes. **Quanto maior o Fisher Score, melhor a feature distingue as classes do ALVO**.
3. **Mutual Information** (MI) $\rightarrow$  Quantifica a dependência estatística entre a feature e o ALVO. Ao contrário de correlações lineares, a MI consegue capturar relações não-lineares. **Valores altos de Mutual Information indicam que a feature compartilha mais informações com o ALVO, tornando-a mais relevante para a predição**.

# Output do notebook

* Este output entrega além das métricas acima para cada variável, também agrega um sumário estatístico das variáveis numéricas que inclui o percentual de missing.   

In [1]:
#Imports
import pandas as pd
import numpy as np
#Pacotes estatísticos
from scipy import stats #Estatística KS
from sklearn.feature_selection import mutual_info_classif #Mutual Information
from sklearn.feature_selection import SelectKBest, f_classif #Fisher Score
##Eliminar os warnings
import warnings
warnings.filterwarnings("ignore")
##Ver todas as colunas do data frame
pd.set_option('display.max_columns', None)
##Ver todas as linhas do data frame
pd.set_option('display.max_rows', None)

### Fómulas do KS2

In [2]:
#KS2
def calcular_ks_2samp(df, alvo, escore):
    bons_escore = df.loc[df[alvo] == 0, escore].rename('bons')
    maus_escore = df.loc[df[alvo] == 1, escore].rename('maus')
    return stats.ks_2samp(bons_escore, maus_escore).statistic

# 1 - Importação dos dados

In [3]:
#Ver a codificação do arquivo
import pandas as pd

#Tentar as codificações mais comuns para arquivos em português
codificacoes = ['latin-1', 'iso-8859-1', 'cp1252', 'utf-8']

for encoding in codificacoes:
    try:
        df = pd.read_csv('dados_credit.csv', sep=';', encoding=encoding)
        print(f"✅ Arquivo lido com sucesso usando codificação: {encoding}")
        print(f"📊 Shape do DataFrame: {df.shape}")
        break
    except UnicodeDecodeError:
        print(f"❌ Falha com {encoding}")
        continue
    except Exception as e:
        print(f"⚠️  Outro erro com {encoding}: {e}")
        continue
else:
    print("❌ Nenhuma codificação funcionou")

✅ Arquivo lido com sucesso usando codificação: latin-1
📊 Shape do DataFrame: (9994, 24)


In [4]:
#Base de dados
df = pd.read_csv('dados_credit.csv', sep = ';', encoding='latin-1')
#Visualização
df.head(2)

,observação,idade,uf,regiao,carencia,PERCENTUAL_ENTRADA_VEICU,ANO_MODELO_VEIC,Idade_Bem,QTDE_PRESTACOES,VLR_PRINCIPAL,TAXA_MENSAL,VLR_RENDA,VLR_MAX_RENDA_PRESUM,PC_COMPR_RENDA_FAMILIAR,Autonomos,Taxistas,Status_prop,score_sis,var_1,var_2,var_3,var_4,var_5,target
0,obs1,62,RJ,SULDESTE,30,0.00,2018,>= 3 anos,60,31537.51,1.77,3690.0,1599.0,24,N,S,EF,664,NaN,NaN,NaN,NaN,S,0.0
1,obs2,75,AP,NORTE,30,62.57,2021,0 Km,12,25654.47,0.00,16400.0,11767.0,13,N,N,EF,673,3.0,A1,RUIM,A1,S,0.0


In [5]:
#Padronização - Colocar todas as colunas em minúsculos
df.columns = df.columns.str.lower()

In [6]:
#Visualização
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   observação                9994 non-null   object 
 1   idade                     9994 non-null   int64  
 2   uf                        9993 non-null   object 
 3   regiao                    9993 non-null   object 
 4   carencia                  9994 non-null   int64  
 5   percentual_entrada_veicu  9994 non-null   float64
 6   ano_modelo_veic           9994 non-null   int64  
 7   idade_bem                 9994 non-null   object 
 8   qtde_prestacoes           9994 non-null   int64  
 9   vlr_principal             9994 non-null   float64
 10  taxa_mensal               9994 non-null   float64
 11  vlr_renda                 9994 non-null   float64
 12  vlr_max_renda_presum      9994 non-null   float64
 13  pc_compr_renda_familiar   9994 non-null   int64  
 14  autonomo

## Criar a variável idade_veiculo

* A partir da variável ano_modelo_veic, eu vou criar a variável chamada **idade_veiculo**. A referência é o ano de 2021.

In [7]:
df['idade_veiculo'] = 2021 - df['ano_modelo_veic']

## Remover as linhas que não tem informação do ALVO (target)

In [8]:
df.dropna(subset=['target'], inplace=True)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 9491 entries, 0 to 9978
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   observação                9491 non-null   object 
 1   idade                     9491 non-null   int64  
 2   uf                        9491 non-null   object 
 3   regiao                    9491 non-null   object 
 4   carencia                  9491 non-null   int64  
 5   percentual_entrada_veicu  9491 non-null   float64
 6   ano_modelo_veic           9491 non-null   int64  
 7   idade_bem                 9491 non-null   object 
 8   qtde_prestacoes           9491 non-null   int64  
 9   vlr_principal             9491 non-null   float64
 10  taxa_mensal               9491 non-null   float64
 11  vlr_renda                 9491 non-null   float64
 12  vlr_max_renda_presum      9491 non-null   float64
 13  pc_compr_renda_familiar   9491 non-null   int64  
 14  autonomo

# 2 - Filtrar as colunas numéricas

In [10]:
#Seleção das colunas numéricas
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns #Filtrar as colunas cujo tipo é inteiro ou float
#Data frame das variáveis numéricas
df_2 = df[numeric_cols]
#Lista de colunas a serem removidas 
cols_to_drop = ['ano_modelo_veic'] 
#Remover as colunas
df_selecao_inicial = df_2.drop(columns=cols_to_drop)
#Visualização do data frame das colunas numéricas
df_selecao_inicial.head(3)

,idade,carencia,percentual_entrada_veicu,qtde_prestacoes,vlr_principal,taxa_mensal,vlr_renda,vlr_max_renda_presum,pc_compr_renda_familiar,score_sis,var_1,target,idade_veiculo
0,62,30,0.00,60,31537.51,1.77,3690.0,1599.0,24,664,NaN,0.0,3
1,75,30,62.57,12,25654.47,0.00,16400.0,11767.0,13,673,3.0,0.0,0
2,56,30,53.00,30,29476.83,0.00,9840.0,2665.0,10,760,3.0,0.0,0


In [11]:
#Total de colunas numéricas (incluindo o ALVO)
df_selecao_inicial.shape[1] #13

13

## Partição do data frame

In [12]:
#Data frame das features
features_quantitativas = df_selecao_inicial.drop(columns=['target']) # Todas as colunas exceto 'ALVO'
#Data frame do ALVO
y = df_selecao_inicial['target'] 

# 3- Estatística descritiva das features

## Estatísticas calculadas
* Média;
* Mediana;
* Máximo;
* Mínimo;
* Percentil 25 (P_25);
* Percentil 75 (P_75);
* Desvio Padrão;
* Coeficiente de Variação.

In [13]:
estatisticas = {
    'MEDIA': features_quantitativas.mean(),
    'MEDIANA': features_quantitativas.median(),
    'MAXIMO': features_quantitativas.max(),
    'MINIMO': features_quantitativas.min(),
    'P_25': features_quantitativas.quantile(0.25),
    'P_75': features_quantitativas.quantile(0.75),
    'DESVIO_PADRAO': features_quantitativas.std(),
    'COEFICIENTE_VARIACAO': (features_quantitativas.std() / features_quantitativas.mean()) * 100
}

# 2. Criar DataFrame e formatar nomes das variáveis em MAIÚSCULAS
df_estatisticas = pd.DataFrame(estatisticas)
df_estatisticas.index = df_estatisticas.index.str.upper()  # Nomes das variáveis em maiúsculas

# 3. Resetar índice para transformar 'VARIAVEL' em coluna
df_estatisticas = df_estatisticas.reset_index().rename(columns={'index': 'VARIAVEL'})

# 4. Ordenar colunas (sem indentação inesperada!)
colunas_ordenadas = ['VARIAVEL', 'MEDIA', 'MEDIANA', 'DESVIO_PADRAO', 'COEFICIENTE_VARIACAO',
                     'MINIMO', 'P_25', 'P_75', 'MAXIMO']  # <-- Sem espaços/tabs antes desta linha
df_estatisticas = df_estatisticas[colunas_ordenadas]

# 5. Mostrar resultado
df_estatisticas.head(3)

,VARIAVEL,MEDIA,MEDIANA,DESVIO_PADRAO,COEFICIENTE_VARIACAO,MINIMO,P_25,P_75,MAXIMO
0,IDADE,49.636287,49.00,13.740468,27.682305,20.0,39.00,60.000,91.00
1,CARENCIA,30.287641,30.00,3.260950,10.766604,30.0,30.00,30.000,90.00
2,PERCENTUAL_ENTRADA_VEICU,38.511558,37.19,22.042758,57.236733,0.0,23.53,54.905,95.19


# 4 - KS das variáveis

* **1** - Criar uma lista com as colunas numéricas.

In [14]:
#Criar uma lista com o nome das variáveis sem o ALVO
colunas_var_quant = list(features_quantitativas.columns.values) 

* **2** - Calcular o KS2 para cada variável.

In [16]:
#Criação de um dicionário
dicionario = {}
#Aplicação de um laço para o cálculo do KS2 da lista de variáveis em colunas_var_quant
for coluna in colunas_var_quant:
   resultado = calcular_ks_2samp(df_selecao_inicial,'target', coluna)
   dicionario[coluna] = resultado

* **3** - Converter este dicionário num data frame.

In [17]:
# 1. Converter o dicionário em DataFrame
df_ks2 = pd.DataFrame(list(dicionario.items()), columns=['VARIAVEL', 'KS2'])

# 2. Ajustar KS2: multiplicar por 100 e arredondar para 2 casas decimais
df_ks2['KS2'] = round(df_ks2['KS2'] * 100, 2)

# 3. Converter os VALORES da coluna 'VARIAVEL' para MAIÚSCULAS
df_ks2['VARIAVEL'] = df_ks2['VARIAVEL'].str.upper()  # Esta linha faz a conversão

# 4. Visualizar as 3 primeiras linhas
df_ks2.head(3)

,VARIAVEL,KS2
0,IDADE,14.95
1,CARENCIA,0.87
2,PERCENTUAL_ENTRADA_VEICU,37.38


# 5 - Fisher score das variáveis

In [ ]:
# 1. Calcular percentual de missings e identificar constantes
missing_percent = features_quantitativas.isna().mean() * 100
constant_cols = features_quantitativas.columns[features_quantitativas.nunique() == 1].tolist()  # Colunas com todos valores iguais

# 2. Inicializar array de Fisher Scores com zeros
fisher_scores = np.zeros(len(features_quantitativas.columns))

# 3. Identificar colunas para cálculo (não constantes e missing < 100%)
cols_validas = [
    col for col in features_quantitativas.columns 
    if (missing_percent[col] < 100) and (col not in constant_cols)
]

if cols_validas:
    # Remover linhas com NaN e alinhar target
    features_validas = features_quantitativas[cols_validas].dropna()
    y_valid = y.loc[features_validas.index]
    
    # Calcular Fisher Score e substituir NaN por 0
    f_scores, _ = f_classif(features_validas, y_valid)
    f_scores = np.nan_to_num(f_scores, nan=0.0)  # Garante NaN → 0
    
    # Atribuir valores às posições corretas
    idx_validas = [features_quantitativas.columns.get_loc(col) for col in cols_validas]
    fisher_scores[idx_validas] = f_scores

# 4. Criar DataFrame final
df_fisher = pd.DataFrame({
    'VARIAVEL': features_quantitativas.columns.str.upper(),
    'FISHER_SCORE': fisher_scores
}).sort_values('FISHER_SCORE', ascending=False)

print("\nRESULTADO DO FISHER SCORE (SCORE=0 PARA MISSING/CONSTANTES):")
display(df_fisher.head(10).style.format({'FISHER_SCORE': '{:.2f}'}))

# 6 - Mutual Information e percentual de missing das variáveis

In [ ]:
# 1. Identificar colunas com 100% missings
missing_percent = features_quantitativas.isna().mean() * 100
cols_100p_missing = missing_percent[missing_percent == 100].index.tolist()

# 2. Inicializar um array de zeros para os resultados
mi_valores = np.zeros(len(features_quantitativas.columns))

# 3. Calcular MI apenas para colunas com dados válidos (<100% missings)
cols_validas = [col for col in features_quantitativas.columns if col not in cols_100p_missing]

if cols_validas:  # Se houver colunas válidas
    # Remover linhas com NaN apenas nas colunas válidas
    features_validas = features_quantitativas[cols_validas].dropna()
    
    # Verificar se ainda há dados após remoção de NaN
    if len(features_validas) > 0:
        y_valid = y.loc[features_validas.index]  # Ajustar o target
        
        # Calcular MI com random_state para reprodutibilidade
        mi_valores_validos = mutual_info_classif(
            features_validas, 
            y_valid, 
            random_state=42  # Garante resultados consistentes
        )
        
        # Atribuir os valores calculados
        idx_validas = [features_quantitativas.columns.get_loc(col) for col in cols_validas]
        mi_valores[idx_validas] = mi_valores_validos
    else:
        print("Aviso: Todas as linhas foram removidas ao eliminar NaN nas colunas válidas.")

# 4. Criar DataFrame final
df_mi = pd.DataFrame({
    'VARIAVEL': features_quantitativas.columns.str.upper(),  # Nomes em maiúsculas
    'MI_VALOR': mi_valores,
    'PERCENTUAL_MISSING': missing_percent.values
}).sort_values('MI_VALOR', ascending=False)

# 5. Visualização com formatação
print("\nRESULTADO DA SELEÇÃO DE VARIÁVEIS POR MUTUAL INFORMATION:")
display(df_mi.head(3).style.format({
    'MI_VALOR': '{:.4f}',
    'PERCENTUAL_MISSING': '{:.1f}'
}))

# 7 - Junção dos resultados numa única tabela

In [ ]:
df_final = (
    df_estatisticas
    .merge(df_ks2, on='VARIAVEL', how='left')      # Left join com df_ks2
    .merge(df_fisher, on='VARIAVEL', how='left')   # Left join com df_fisher
    .merge(df_mi, on='VARIAVEL', how='left')       # Left join com df_mi
)
#Informações do data frame final
df_final.info()

In [ ]:
#Visualização
df_final.head(3)

In [ ]:
#Exportar para CSV
df_final.to_csv('eda_selecao_varaiveis.csv', header=True, index=False, sep=';')